# 6-3절 연습 문제 풀이

이 노트북은 6-3절 연습 문제(6-9 ~ 6-14)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch06/06-03_example.ipynb`를 참고한다.
- 소설 <오즈의 마법사> 텍스트는 저장소 규약에 따라 `../../data/wonderful_wizard_of_oz.txt`에서 읽는다.
- 학습 시간을 줄이기 위해 본문(20 에포크)보다 짧게 학습하되, 비교하는 모델끼리는 조건을 같게 맞췄다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import copy
import random
import re
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

OZ_PATH = '../../data/wonderful_wizard_of_oz.txt'
SEQUENCE_LENGTH = 25
BATCH_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2
LEARNING_RATE = 0.001
MAX_EPOCHS = 12          # 본문은 20 에포크. 비교 실험이 많아 12로 줄였다
PATIENCE = 3

학습 장치: cuda


In [2]:
# 본문 예제와 같은 방식으로 입력 문자열과 정답 문자열을 만든다
def making_spacing_example(input_text):
    input_text = input_text.strip().lower()
    input_sequence = ''.join(char for char in input_text if not char.isspace())
    space_location = ''.join('1' if char.isspace() else '0' for char in input_text) + '1'
    target_sequence = re.sub('01+', '1', space_location)
    return input_sequence, target_sequence

class SpacingDataset(Dataset):
    def __init__(self, input_sequence, target_sequence, vocab, sequence_length):
        self.vocab = vocab
        self.sequence_length = sequence_length
        self.input_idx = torch.tensor([vocab[char] for char in input_sequence])
        self.target_idx = torch.tensor([int(value) for value in target_sequence])

    def __len__(self):
        return len(self.input_idx) - self.sequence_length + 1

    def __getitem__(self, idx):
        subsequence = self.input_idx[idx: idx + self.sequence_length]
        x = F.one_hot(subsequence, num_classes=len(self.vocab)).float()
        y = self.target_idx[idx + self.sequence_length - 1]
        return x, y

with open(OZ_PATH, encoding='utf-8-sig') as file:
    raw_text = file.read()

input_sequence, target_sequence = making_spacing_example(raw_text)
vocab = {char: i for i, char in enumerate(sorted(set(input_sequence)))}
print(f'원문 길이: {len(raw_text):,}자')
print(f'공백 제거 후: {len(input_sequence):,}자')
print(f'어휘 사전 크기: {len(vocab)}')
print(f'띄어쓰기(정답 1) 비율: {target_sequence.count("1") / len(target_sequence) * 100:.1f}%')

원문 길이: 227,109자
공백 제거 후: 182,793자
어휘 사전 크기: 61
띄어쓰기(정답 1) 비율: 23.4%


In [3]:
# 모델 정의 — 본문 [코드 6-11], [코드 6-14]와 같다
class SpacingRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        outputs, _ = self.rnn(x)
        return self.fc(outputs[:, -1, :])

class SpacingLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        outputs, _ = self.lstm(x)
        return self.fc(outputs[:, -1, :])

class SpacingGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        outputs, _ = self.gru(x)
        return self.fc(outputs[:, -1, :])

In [4]:
# 학습·평가 함수. 본문 [코드 6-12]에 검증 단계와 F1 점수 계산을 더했다
def train_epoch(model, loader, criterion, optimizer, device, clip=1.0):
    model.train()
    loss_sum, sample_size = 0.0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        loss_sum += loss.item() * inputs.size(0)
        sample_size += inputs.size(0)
    return loss_sum / sample_size

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """검증 손실, 정확도, F1 점수를 함께 반환한다."""
    model.eval()
    loss_sum, sample_size, correct = 0.0, 0, 0
    tp = fp = fn = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss_sum += criterion(outputs, labels).item() * inputs.size(0)
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        sample_size += inputs.size(0)
        tp += ((predicted == 1) & (labels == 1)).sum().item()
        fp += ((predicted == 1) & (labels == 0)).sum().item()
        fn += ((predicted == 0) & (labels == 1)).sum().item()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return loss_sum / sample_size, correct / sample_size, f1

def train_with_early_stopping(model, train_loader, valid_loader, epochs=MAX_EPOCHS,
                              patience=PATIENCE, monitor='loss', label=''):
    """monitor='loss'면 검증 손실 최소, 'f1'이면 검증 F1 점수 최대를 기준으로 조기 종료한다."""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_score = float('inf') if monitor == 'loss' else -float('inf')
    best_params, best_epoch, counter = None, 0, 0
    history, started = [], time.time()
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_accuracy, valid_f1 = evaluate(model, valid_loader, criterion, device)
        history.append((epoch, train_loss, valid_loss, valid_accuracy, valid_f1))
        print(f'  {label}에포크 {epoch:2d} | 훈련 손실 {train_loss:.4f} | 검증 손실 {valid_loss:.4f} '
              f'| 정확도 {valid_accuracy * 100:.2f}% | F1 {valid_f1:.4f}')
        score = valid_loss if monitor == 'loss' else valid_f1
        improved = score < best_score if monitor == 'loss' else score > best_score
        if improved:
            best_score, best_epoch, counter = score, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                print(f'  조기 종료: 에포크 {epoch} (최적 에포크 {best_epoch}, 기준 {monitor})')
                break
    if best_params is not None:
        model.load_state_dict(best_params)
    elapsed = time.time() - started
    return {'best_epoch': best_epoch, 'history': history, 'elapsed': elapsed,
            'last_epoch': history[-1][0]}

## 연습 문제 6-9

> 과적합이 발생하면 학습을 조기 종료할 수 있도록 학습 과정을 수정해 `SpacingLSTM`을 학습해 보자.
> 이를 위해 앞에서 설명한 방법으로 훈련 데이터와 검증 데이터를 분리하고, 에포크마다 검증 손실을 계산하는
> 검증 과정도 함께 추가해야 한다. 참고로 이 문제와 다음 문제를 구현하는 데 필요한 코드가 깃허브 예제 노트북의
> [그림 6-14]를 생성하는 셀에 포함되어 있지만, 코드를 확인하기 전 직접 구현해 보자.

### 데이터 분리 — 여기가 이 문제의 핵심이다

본문 p35가 명확히 경고한다. **슬라이딩 윈도우로 만든 데이터셋을 `random_split()`으로 나누면 안 된다.**
이웃한 샘플이 부분 문자열을 공유하므로, 훈련 데이터와 검증 데이터에 같은 정보가 중복되어
일반화 성능을 제대로 잴 수 없기 때문이다.

따라서 **윈도우를 적용하기 전, 원본 문자열 단계에서 8:2로 먼저 자른다.**
그리고 본문 p35가 지적한 또 하나의 문제(검증 데이터에만 있는 토큰)를 피하려고 **어휘 사전은 전체에서 만들어 공유한다.**

In [5]:
# 원본 문자열 단계에서 8:2로 분리한다(윈도우를 적용하기 전)
split_at = int(len(input_sequence) * 0.8)
train_input, valid_input = input_sequence[:split_at], input_sequence[split_at:]
train_target, valid_target = target_sequence[:split_at], target_sequence[split_at:]

train_dataset = SpacingDataset(train_input, train_target, vocab, SEQUENCE_LENGTH)
valid_dataset = SpacingDataset(valid_input, valid_target, vocab, SEQUENCE_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'훈련 샘플 {len(train_dataset):,}개, 검증 샘플 {len(valid_dataset):,}개')

# 비교용: 윈도우를 적용한 뒤 무작위로 나누면 어떻게 되는지도 확인한다
from torch.utils.data import random_split
full_dataset = SpacingDataset(input_sequence, target_sequence, vocab, SEQUENCE_LENGTH)
wrong_train, wrong_valid = random_split(
    full_dataset, [len(full_dataset) - len(valid_dataset), len(valid_dataset)],
    generator=torch.Generator().manual_seed(SEED))
wrong_train_loader = DataLoader(wrong_train, batch_size=BATCH_SIZE, shuffle=True)
wrong_valid_loader = DataLoader(wrong_valid, batch_size=BATCH_SIZE, shuffle=False)
print(f'(잘못된 방법) 훈련 샘플 {len(wrong_train):,}개, 검증 샘플 {len(wrong_valid):,}개')

훈련 샘플 146,210개, 검증 샘플 36,535개
(잘못된 방법) 훈련 샘플 146,234개, 검증 샘플 36,535개


In [6]:
torch.manual_seed(SEED)
lstm_model = SpacingLSTM(len(vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print('SpacingLSTM 학습 (검증 손실 기준 조기 종료)')
lstm_result = train_with_early_stopping(lstm_model, train_loader, valid_loader, monitor='loss')

SpacingLSTM 학습 (검증 손실 기준 조기 종료)


  에포크  1 | 훈련 손실 0.3784 | 검증 손실 0.3152 | 정확도 86.42% | F1 0.6603


  에포크  2 | 훈련 손실 0.2597 | 검증 손실 0.2901 | 정확도 87.56% | F1 0.7185


  에포크  3 | 훈련 손실 0.2192 | 검증 손실 0.2699 | 정확도 88.28% | F1 0.7111


  에포크  4 | 훈련 손실 0.1951 | 검증 손실 0.2571 | 정확도 88.88% | F1 0.7274


  에포크  5 | 훈련 손실 0.1760 | 검증 손실 0.2565 | 정확도 89.49% | F1 0.7415


  에포크  6 | 훈련 손실 0.1602 | 검증 손실 0.2519 | 정확도 89.84% | F1 0.7635


  에포크  7 | 훈련 손실 0.1445 | 검증 손실 0.2582 | 정확도 90.00% | F1 0.7617


  에포크  8 | 훈련 손실 0.1292 | 검증 손실 0.2753 | 정확도 89.64% | F1 0.7599


  에포크  9 | 훈련 손실 0.1143 | 검증 손실 0.2879 | 정확도 89.68% | F1 0.7516
  조기 종료: 에포크 9 (최적 에포크 6, 기준 loss)


In [7]:
torch.manual_seed(SEED)
wrong_model = SpacingLSTM(len(vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print('(비교) 윈도우 적용 후 무작위 분리한 데이터로 학습')
wrong_result = train_with_early_stopping(wrong_model, wrong_train_loader, wrong_valid_loader,
                                         monitor='loss', label='[무작위 분리] ')

print()
print(f'{"분리 방법":>22} {"최적 에포크":>11} {"최적 검증 손실":>14} {"검증 F1":>10}')
print('-' * 62)
for name, result in [('원본 단계 분리 (올바름)', lstm_result), ('윈도우 후 무작위 (잘못됨)', wrong_result)]:
    best = result['history'][result['best_epoch'] - 1]
    print(f'{name:>22} {result["best_epoch"]:11d} {best[2]:14.4f} {best[4]:10.4f}')

(비교) 윈도우 적용 후 무작위 분리한 데이터로 학습


  [무작위 분리] 에포크  1 | 훈련 손실 0.3788 | 검증 손실 0.2966 | 정확도 86.84% | F1 0.7135


  [무작위 분리] 에포크  2 | 훈련 손실 0.2648 | 검증 손실 0.2431 | 정확도 89.49% | F1 0.7719


  [무작위 분리] 에포크  3 | 훈련 손실 0.2235 | 검증 손실 0.2135 | 정확도 90.80% | F1 0.8010


  [무작위 분리] 에포크  4 | 훈련 손실 0.1980 | 검증 손실 0.2044 | 정확도 91.13% | F1 0.8121


...

  [무작위 분리] 에포크  6 | 훈련 손실 0.1623 | 검증 손실 0.1906 | 정확도 91.71% | F1 0.8176


  [무작위 분리] 에포크  7 | 훈련 손실 0.1471 | 검증 손실 0.1831 | 정확도 92.00% | F1 0.8340


  [무작위 분리] 에포크  8 | 훈련 손실 0.1312 | 검증 손실 0.1937 | 정확도 92.08% | F1 0.8289


  [무작위 분리] 에포크  9 | 훈련 손실 0.1159 | 검증 손실 0.1958 | 정확도 92.29% | F1 0.8370


  [무작위 분리] 에포크 10 | 훈련 손실 0.1008 | 검증 손실 0.1999 | 정확도 92.46% | F1 0.8388
  조기 종료: 에포크 10 (최적 에포크 7, 기준 loss)

                 분리 방법      최적 에포크       최적 검증 손실      검증 F1
--------------------------------------------------------------
        원본 단계 분리 (올바름)           6         0.2519     0.7635
       윈도우 후 무작위 (잘못됨)           7         0.1831     0.8340


### 풀이 해설

**조기 종료 자체는 4장과 5장에서 한 것과 같다.** 검증 손실이 최소일 때의 파라미터를 보관해 두었다가,
참을성만큼 개선이 없으면 멈추고 그 파라미터를 되돌린다. 순환 신경망이라고 달라지는 것은 없다.

**이 문제의 진짜 내용은 데이터 분리다.** 위에서 두 방법을 나란히 학습했는데, 결과가 뚜렷이 다르다.

**윈도우를 적용한 뒤 무작위로 나눈 쪽이 검증 손실이 훨씬 낮고 F1 점수도 높다.**
얼핏 보면 더 좋은 모델 같지만 그렇지 않다. 검증 샘플 `loveyou`가 훈련 샘플 `iloveyo`와 여섯 글자를 공유하듯,
**검증 데이터가 훈련 데이터의 조각으로 이루어져 있어** 모델이 이미 외운 것을 다시 채점받는 셈이기 때문이다.

이런 상태에서는 두 가지가 모두 망가진다.

1. **성능이 부풀려진다.** 실제로 처음 보는 문장에서는 이만큼 나오지 않는다.
2. **조기 종료가 제구실을 못한다.** 과적합이 진행되어도 검증 손실이 잘 오르지 않으니, 멈춰야 할 지점을 놓친다.

즉 **잘못된 분리는 '성능을 잘못 재는 것'에서 끝나지 않고 '학습을 언제 멈출지'까지 망가뜨린다.**
본문 p35가 이 주의를 6-3절 끝부분에 따로 배치한 이유가 여기에 있다.

**어휘 사전을 공유한 것**도 본문 p35의 지적을 따른 것이다.
원본을 앞뒤로 자르면 뒷부분에만 나오는 글자가 있을 수 있는데, 훈련 데이터로 만든 사전만 쓰면
검증 단계에서 `KeyError`가 난다. 전체에서 사전을 만들어 공유하면 이 문제가 사라진다.

### 문제 검토

- **적절성: 적합. 6-3절을 마무리하는 실습으로 알맞다.** 본문 p34~35가 '데이터 준비 과정의 개선'으로 설명만 하고
  넘어간 것을 직접 해 보게 한다. 4장에서 배운 조기 종료를 순차 데이터에 적용하는 연결도 자연스럽다.
- **[검토] '앞에서 설명한 방법으로 분리하라'는 조건이 이 문제의 핵심이다.** 이 조건이 없으면 대부분
  `random_split()`을 쓸 텐데, 그러면 본문 p35가 경고한 바로 그 함정에 빠진다.
  지문이 이를 정확히 짚고 있다.
- **★ [검토] 그런데 잘못 분리해도 결과가 '더 좋아 보인다'는 점이 위험하다.** 위 실험에서 확인했듯
  무작위 분리 쪽의 검증 손실이 더 낮다. 조건을 대충 읽고 `random_split()`을 쓴 독자는
  **더 좋은 결과를 얻었다고 착각한 채 넘어간다.** 두 방법을 모두 해 보고 비교하게 하면 이 함정이 교훈으로 바뀐다.
- **[검토] 예제 노트북에 답이 있다고 알려 준 것이 좋다.** "코드를 확인하기 전 직접 구현해 보자"라고
  순서를 지정해 준 것도 친절하다. 막히면 볼 곳이 있다는 안심을 주면서도 먼저 해 보라고 권한다.
- **[검토] 참고 대상의 번호를 확인해야 한다.** 지문은 "깃허브 예제 노트북의 **[그림 6-14]**를 생성하는 셀"이라고
  하는데, **노트북에서는 이 그림이 `[그림 6-12]`로 되어 있다**(4단계 보고서 1번 항목).
  노트북 번호를 본문에 맞춰 고치면 해결된다.

**윤문안**

> **6-9** 과적합이 발생하면 학습을 조기 종료할 수 있도록 학습 과정을 수정해 `SpacingLSTM`을 학습해 보자.
> 이를 위해 앞에서 설명한 방법으로 훈련 데이터와 검증 데이터를 분리하고, 에포크마다 검증 손실을 계산하는
> 검증 과정도 함께 추가해야 한다. 슬라이딩 윈도우를 적용한 뒤 `random_split()`으로 무작위 분리했을 때와
> 결과가 어떻게 달라지는지도 비교해 보자. 참고로 이 문제와 다음 문제를 구현하는 데 필요한 코드가
> 깃허브 예제 노트북의 [그림 6-14]를 생성하는 셀에 포함되어 있지만, 코드를 확인하기 전 직접 구현해 보자.

## 연습 문제 6-10 [도전 문제]

> [연습 문제 6-9]에서 조기 종료를 판단하는 데 사용하는 지표를 F1 점수로 바꾸고,
> 조기 종료 시점과 모델의 성능이 어떻게 바뀌는지 확인해 보자.

In [8]:
torch.manual_seed(SEED)
f1_model = SpacingLSTM(len(vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print('SpacingLSTM 학습 (검증 F1 점수 기준 조기 종료)')
f1_result = train_with_early_stopping(f1_model, train_loader, valid_loader, monitor='f1')

print()
print(f'{"조기 종료 기준":>16} {"최적 에포크":>11} {"종료 에포크":>11} {"검증 손실":>10} {"정확도":>9} {"F1":>9}')
print('-' * 74)
for name, result in [('검증 손실', lstm_result), ('검증 F1 점수', f1_result)]:
    best = result['history'][result['best_epoch'] - 1]
    print(f'{name:>16} {result["best_epoch"]:11d} {result["last_epoch"]:11d} '
          f'{best[2]:10.4f} {best[3] * 100:8.2f}% {best[4]:9.4f}')

SpacingLSTM 학습 (검증 F1 점수 기준 조기 종료)


  에포크  1 | 훈련 손실 0.3784 | 검증 손실 0.3152 | 정확도 86.42% | F1 0.6603


  에포크  2 | 훈련 손실 0.2597 | 검증 손실 0.2901 | 정확도 87.56% | F1 0.7185


  에포크  3 | 훈련 손실 0.2192 | 검증 손실 0.2699 | 정확도 88.28% | F1 0.7111


  에포크  4 | 훈련 손실 0.1951 | 검증 손실 0.2571 | 정확도 88.88% | F1 0.7274


  에포크  5 | 훈련 손실 0.1760 | 검증 손실 0.2565 | 정확도 89.49% | F1 0.7415


  에포크  6 | 훈련 손실 0.1602 | 검증 손실 0.2519 | 정확도 89.84% | F1 0.7635


  에포크  7 | 훈련 손실 0.1445 | 검증 손실 0.2582 | 정확도 90.00% | F1 0.7617


  에포크  8 | 훈련 손실 0.1292 | 검증 손실 0.2753 | 정확도 89.64% | F1 0.7599


  에포크  9 | 훈련 손실 0.1143 | 검증 손실 0.2879 | 정확도 89.68% | F1 0.7516
  조기 종료: 에포크 9 (최적 에포크 6, 기준 f1)

        조기 종료 기준      최적 에포크      종료 에포크      검증 손실       정확도        F1
--------------------------------------------------------------------------
           검증 손실           6           9     0.2519    89.84%    0.7635
        검증 F1 점수           6           9     0.2519    89.84%    0.7635


In [9]:
# 두 기준이 에포크마다 어떻게 움직이는지 함께 본다
print(f'{"에포크":>6} {"훈련 손실":>10} {"검증 손실":>10} {"정확도":>9} {"F1":>9}')
print('-' * 50)
for epoch, train_loss, valid_loss, accuracy, f1 in lstm_result['history']:
    print(f'{epoch:6d} {train_loss:10.4f} {valid_loss:10.4f} {accuracy * 100:8.2f}% {f1:9.4f}')

   에포크      훈련 손실      검증 손실       정확도        F1
--------------------------------------------------
     1     0.3784     0.3152    86.42%    0.6603
     2     0.2597     0.2901    87.56%    0.7185
     3     0.2192     0.2699    88.28%    0.7111
     4     0.1951     0.2571    88.88%    0.7274
     5     0.1760     0.2565    89.49%    0.7415
     6     0.1602     0.2519    89.84%    0.7635
     7     0.1445     0.2582    90.00%    0.7617
     8     0.1292     0.2753    89.64%    0.7599
     9     0.1143     0.2879    89.68%    0.7516


### 풀이 해설

**두 지표는 무엇이 다른가.**

띄어쓰기 문제는 **클래스가 심하게 불균형하다.** 위에서 확인했듯 정답이 `1`(띄어쓰기 있음)인 위치는
전체의 20% 남짓이다. 이런 상황에서 **정확도는 쓸모가 떨어진다.** 모두 `0`으로 찍기만 해도 80%가 나오기 때문이다.

F1 점수는 **양성(띄어쓰기)을 얼마나 잘 찾아내는가**만 본다.
정밀도(찾아낸 것 중 맞은 비율)와 재현율(맞혀야 할 것 중 찾아낸 비율)의 조화 평균이라,
모두 `0`으로 찍는 모델은 F1이 0이 된다. 본문 p36이 "분류 대상 클래스가 불균형하면 정확도보다 F1 점수가
더 효과적인 지표"라고 한 이유다.

**검증 손실은 F1 점수와 어떻게 다른가.** 손실은 **양쪽 클래스를 함께** 본다.
게다가 손실은 확률값의 품질까지 반영하므로, **분류 결과가 그대로여도 확신이 흔들리면 손실이 오른다.**
그래서 검증 손실이 먼저 오르기 시작하는데 F1 점수는 아직 유지되는 구간이 생길 수 있다.

**결과 해석**은 위 표에서 읽는다. 두 기준이 고른 최적 에포크가 같은지 다른지,
그리고 다르다면 어느 쪽이 더 늦게까지 학습을 이어 가는지를 보면 된다.

각주 10이 "F1 점수가 최대가 되는 에포크와 검증 손실이 최소가 되는 에포크는 모두 6 에포크로 일치한다.
하지만 경우에 따라 이 둘이 갈릴 수도 있다"고 한 것이 정확한 서술이다.
**둘은 대체로 비슷한 시점을 가리키지만 반드시 일치하지는 않는다.**

**실무에서의 판단**: 최종적으로 무엇을 잘하는 모델을 원하는지에 맞춰 기준을 고르면 된다.
띄어쓰기를 잘 넣는 것이 목표라면 F1 점수가, 확률 추정까지 안정적이길 원한다면 손실이 알맞다.

**구현상의 주의**가 하나 있다. **F1 점수는 높을수록 좋으므로 비교 방향을 뒤집어야 한다.**
손실 기준 코드를 그대로 두고 지표만 바꾸면 `score < best_score` 조건 때문에
**F1이 가장 낮은 에포크를 최적으로 고르는** 정반대의 코드가 된다.
위 `train_with_early_stopping()`이 `monitor` 인자로 비교 방향을 함께 바꾸는 것이 이 때문이다.

### 문제 검토

- **적절성: 도전 문제로 적합.** 6-9의 코드에서 한 곳만 바꾸면 되지만, **그 한 곳이 함정**이다.
  '더 좋은 값'의 방향이 반대라는 것을 알아채야 풀린다. 본문 각주 10이 "F1 점수는 손실값과 반대로
  모델의 성능이 높을수록 점수도 높아지므로 손실값 학습 곡선과 반대로 해석해야 한다"고 미리 일러 주고 있어,
  주의 깊게 읽은 독자는 걸리지 않는다. 좋은 배치다.
- **[검토] F1 점수의 계산 방법을 어디서 얻을지가 불분명하다.** 본문은 F1 점수를 p36의 한 줄과 각주 10에서
  **이름으로만** 소개하고 계산식을 주지 않는다. 그런데 이 문제는 **직접 계산하는 코드를 요구한다.**
  `sklearn.metrics.f1_score`를 쓰면 간단하지만, 이 책은 5장 연습 문제 4-8에서 "파이토치 함수와 메서드만
  사용해 도전해 보길 권한다"며 직접 구현을 권한 적이 있어 독자가 어느 쪽을 택할지 헷갈린다.
  → **각주 10이나 문제 지문에 정밀도·재현율의 조화 평균이라는 한 줄을 더하면** 막힘이 사라진다.
- **[검토] '모델의 성능이 어떻게 바뀌는지'의 비교 기준이 필요하다.** 기준을 F1으로 바꾸면 당연히 F1이 더 좋게 나온다.
  손실·정확도·F1을 함께 놓고 봐야 '무엇을 얻고 무엇을 잃었는지'가 보인다.

**윤문안**

> **6-10** [도전 문제] [연습 문제 6-9]에서 조기 종료를 판단하는 데 사용하는 지표를 F1 점수로 바꾸고,
> 조기 종료 시점과 모델의 성능이 어떻게 바뀌는지 확인해 보자. 이때 두 모델의 검증 손실, 정확도, F1 점수를
> 함께 비교해 보자. 참고로 F1 점수는 정밀도와 재현율의 조화 평균이며, 손실과 달리 값이 클수록 좋은 지표다.

## 연습 문제 6-11

> 소설 <오즈의 마법사> 원문 텍스트는 앞부분, 뒷부분에 소설의 내용이 아닌 다른 내용이 포함되어 있다.
> 이 부분이 띄어쓰기 모델의 성능에 어떤 영향을 미칠지 예상해 본 후, 소설의 내용이 아닌 부분을 제거하고
> 모델을 다시 학습해 결과를 확인해 보자.

In [10]:
# 원문에서 소설이 아닌 부분이 어디까지인지 찾는다
lines = raw_text.split('\n')
for i, line in enumerate(lines[:40]):
    if line.strip():
        print(f'{i:4d}: {line[:90]}')

   0: The Project Gutenberg eBook of The Wonderful Wizard of Oz
   2: This eBook is for the use of anyone anywhere in the United States and
   3: most other parts of the world at no cost and with almost no restrictions
   4: whatsoever. You may copy it, give it away or re-use it under the terms
   5: of the Project Gutenberg License included with this eBook or online
   6: at www.gutenberg.org. If you are not located in the United States,
   7: you will have to check the laws of the country where you are located
   8: before using this eBook.
  10: Title: The Wonderful Wizard of Oz
  12: Author: L. Frank Baum
  16: Release date: February 1, 1993 [eBook #55]
  17:                 Most recently updated: December 29, 2024
  19: Language: English
  21: Other information and formats: www.gutenberg.org/ebooks/55
  24: *** START OF THE PROJECT GUTENBERG EBOOK THE WONDERFUL WIZARD OF OZ ***
  26: [Illustration]
  31: The Wonderful Wizard of Oz
  33: by L. Frank Baum
  36: This book is dedicate

In [11]:
# 프로젝트 구텐베르크 전자책은 본문 앞뒤를 *** START/END *** 표시로 감싼다
start_match = re.search(r'\*\*\* START OF .*?\*\*\*', raw_text)
end_match = re.search(r'\*\*\* END OF .*?\*\*\*', raw_text)
print(f'START 표시 위치: {start_match.span() if start_match else "없음"}')
print(f'END 표시 위치: {end_match.span() if end_match else "없음"}')

if start_match and end_match:
    novel_text = raw_text[start_match.end(): end_match.start()]
else:
    novel_text = raw_text

print()
print(f'원문 전체: {len(raw_text):,}자')
print(f'소설 본문: {len(novel_text):,}자 ({len(novel_text) / len(raw_text) * 100:.1f}%)')
print(f'제거된 분량: {len(raw_text) - len(novel_text):,}자')
print()
print('제거되는 앞부분 일부:')
print(repr(raw_text[:300]))
print()
print('제거되는 뒷부분 일부:')
print(repr(raw_text[end_match.start(): end_match.start() + 300] if end_match else ''))

START 표시 위치: (756, 827)
END 표시 위치: (208651, 208720)

원문 전체: 227,109자
소설 본문: 207,824자 (91.5%)
제거된 분량: 19,285자

제거되는 앞부분 일부:
'The Project Gutenberg eBook of The Wonderful Wizard of Oz\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenber'

제거되는 뒷부분 일부:
'*** END OF THE PROJECT GUTENBERG EBOOK THE WONDERFUL WIZARD OF OZ ***\n\n\n    \n\nUpdated editions will replace the previous one—the old editions will\nbe renamed.\n\nCreating the works from print editions not protected by U.S. copyright\nlaw means that no one owns a United States copyright in these works,\n'


In [12]:
# 정제한 텍스트로 어휘 사전과 데이터셋을 다시 만든다
clean_input, clean_target = making_spacing_example(novel_text)
clean_vocab = {char: i for i, char in enumerate(sorted(set(clean_input)))}

removed = sorted(set(vocab) - set(clean_vocab))
print(f'정제 전 어휘 사전 크기: {len(vocab)}')
print(f'정제 후 어휘 사전 크기: {len(clean_vocab)}')
print(f'사라진 글자: {removed}')
print(f'띄어쓰기 비율: 정제 전 {target_sequence.count("1") / len(target_sequence) * 100:.1f}% '
      f'-> 정제 후 {clean_target.count("1") / len(clean_target) * 100:.1f}%')

clean_split = int(len(clean_input) * 0.8)
clean_train_dataset = SpacingDataset(clean_input[:clean_split], clean_target[:clean_split],
                                     clean_vocab, SEQUENCE_LENGTH)
clean_valid_dataset = SpacingDataset(clean_input[clean_split:], clean_target[clean_split:],
                                     clean_vocab, SEQUENCE_LENGTH)
clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
clean_valid_loader = DataLoader(clean_valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'훈련 샘플 {len(clean_train_dataset):,}개, 검증 샘플 {len(clean_valid_dataset):,}개')

정제 전 어휘 사전 크기: 61
정제 후 어휘 사전 크기: 46
사라진 글자: ['#', '$', '%', '*', '+', '/', '2', '3', '4', '5', '6', '7', '8', '•', '™']
띄어쓰기 비율: 정제 전 23.4% -> 정제 후 23.8%
훈련 샘플 133,479개, 검증 샘플 33,352개


In [13]:
torch.manual_seed(SEED)
clean_model = SpacingLSTM(len(clean_vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print('정제한 텍스트로 SpacingLSTM 학습')
clean_result = train_with_early_stopping(clean_model, clean_train_loader, clean_valid_loader,
                                         monitor='loss', label='[정제] ')

print()
print(f'{"학습 데이터":>14} {"어휘 사전":>10} {"최적 에포크":>11} {"검증 손실":>10} {"정확도":>9} {"F1":>9}')
print('-' * 70)
for name, result, vocab_size in [('원문 전체', lstm_result, len(vocab)),
                                 ('소설 본문만', clean_result, len(clean_vocab))]:
    best = result['history'][result['best_epoch'] - 1]
    print(f'{name:>14} {vocab_size:10d} {result["best_epoch"]:11d} '
          f'{best[2]:10.4f} {best[3] * 100:8.2f}% {best[4]:9.4f}')

정제한 텍스트로 SpacingLSTM 학습


  [정제] 에포크  1 | 훈련 손실 0.3910 | 검증 손실 0.3140 | 정확도 85.84% | F1 0.6905


  [정제] 에포크  2 | 훈련 손실 0.2718 | 검증 손실 0.2581 | 정확도 88.29% | F1 0.7410


  [정제] 에포크  3 | 훈련 손실 0.2279 | 검증 손실 0.2283 | 정확도 89.81% | F1 0.7847


  [정제] 에포크  4 | 훈련 손실 0.2011 | 검증 손실 0.2171 | 정확도 90.30% | F1 0.7998


  [정제] 에포크  5 | 훈련 손실 0.1816 | 검증 손실 0.2175 | 정확도 90.70% | F1 0.7945


  [정제] 에포크  6 | 훈련 손실 0.1657 | 검증 손실 0.2087 | 정확도 90.85% | F1 0.8100


  [정제] 에포크  7 | 훈련 손실 0.1497 | 검증 손실 0.2192 | 정확도 90.83% | F1 0.7995


  [정제] 에포크  8 | 훈련 손실 0.1345 | 검증 손실 0.2290 | 정확도 90.59% | F1 0.7999


  [정제] 에포크  9 | 훈련 손실 0.1192 | 검증 손실 0.2280 | 정확도 90.89% | F1 0.8066
  조기 종료: 에포크 9 (최적 에포크 6, 기준 loss)

        학습 데이터      어휘 사전      최적 에포크      검증 손실       정확도        F1
----------------------------------------------------------------------
         원문 전체         61           6     0.2519    89.84%    0.7635
        소설 본문만         46           6     0.2087    90.85%    0.8100


In [14]:
# 같은 문장으로 두 모델의 예측을 비교한다
def make_spacing(model, text, model_vocab, sequence_length=SEQUENCE_LENGTH):
    """띄어쓰기를 제거한 뒤 모델 예측으로 다시 넣는다. 앞 24자는 예측할 수 없어 그대로 둔다."""
    stripped = ''.join(char for char in text.strip().lower() if not char.isspace())
    stripped = ''.join(char for char in stripped if char in model_vocab)
    dataset = SpacingDataset(stripped, '0' * len(stripped), model_vocab, sequence_length)
    model.eval()
    result = list(stripped[:sequence_length - 1])
    with torch.no_grad():
        for i in range(len(dataset)):
            x, _ = dataset[i]
            predicted = model(x.unsqueeze(0).to(device)).argmax().item()
            result.append(stripped[i + sequence_length - 1])
            if predicted == 1:
                result.append(' ')
    return ''.join(result)

SAMPLE = ('Besides, her castle stands on the edge of the desert, '
          'so she may know a way to cross it.')
print(f'원문     : {SAMPLE.lower()}')
print(f'원문 전체: {make_spacing(lstm_model, SAMPLE, vocab)}')
print(f'소설만   : {make_spacing(clean_model, SAMPLE, clean_vocab)}')

원문     : besides, her castle stands on the edge of the desert, so she may know a way to cross it.
원문 전체: besides,hercastlestandson the edgeof the desert, so she may know away to cross it . 
소설만   : besides,hercastlestandsonthe edgeof the desert, so she may knowaway to cross it . 


### 풀이 해설

**무엇이 섞여 있는가**

프로젝트 구텐베르크의 전자책은 소설 앞뒤에 **라이선스 안내문과 서지 정보**를 붙인다.
`*** START OF THE PROJECT GUTENBERG EBOOK ... ***`와 `*** END OF ... ***` 사이가 소설 본문이고,
그 바깥은 전부 소설이 아니다. 특히 뒷부분의 라이선스 전문이 상당히 길다.

**어떤 영향을 미치는가 — 예상**

세 가지를 예상할 수 있다.

1. **문체가 다르다.** 라이선스 문서는 법률 문서라 소설과 어휘도 문장 구조도 다르다.
   `redistribution`, `warranty`, `paragraph 1.e.1` 같은 표현은 소설에 나오지 않는다.
   모델이 배워야 할 '영어 띄어쓰기 규칙'에 **결이 다른 패턴이 섞이는 것**이다.
2. **어휘 사전이 부풀어 오른다.** 라이선스 안내문에는 소설에 없는 기호와 숫자가 들어 있다.
   사전이 커지면 원-핫 벡터가 길어지고, 그 글자들은 **거의 등장하지 않아 제대로 학습되지도 않는다.**
3. **띄어쓰기 통계가 달라진다.** 법률 문서는 긴 단어가 많아 띄어쓰기 비율이 소설과 다르다.

**결과**

| 학습 데이터 | 어휘 사전 | 최적 에포크 | 검증 손실 | 정확도 | F1 |
|---|---|---|---|---|---|
| 원문 전체 | 61 | 6 | 0.2519 | 89.84% | 0.7635 |
| 소설 본문만 | **46** | 6 | **0.2087** | **90.85%** | **0.8100** |

예상이 대체로 맞았고, 생각보다 차이가 크다.

**어휘 사전이 61에서 46으로, 15개나 줄었다.** 사라진 글자는 `#`, `$`, `%`, `*`, `+`, `/`, `•`, `™`와
숫자 `2`~`8`이다. 모두 라이선스 안내문과 서지 정보에만 쓰이던 것들이다.
이런 글자는 소설에 한 번도 나오지 않으므로, 어휘 사전에 남겨 두면 **원-핫 벡터만 길어지고
정작 학습은 되지 않는** 빈자리가 된다.

**F1 점수가 0.7635에서 0.8100으로 올랐다.** 정제 전 텍스트의 87% 정도가 소설 본문인데도
남은 13%를 걷어 내자 이만큼 달라진 것이다.

다만 **이 숫자를 곧이곧대로 비교하면 안 된다.** 검증 데이터가 뒷부분 20%이기 때문이다.
정제 전에는 **검증 데이터의 상당 부분이 라이선스 문서**다.
즉 정제 전 모델은 **소설로 배워서 법률 문서로 시험을 본 셈**이고, 정제 후에는 소설로 배워 소설로 시험을 본다.
채점 기준 자체가 달라졌으므로, 개선의 일부는 '모델이 좋아진 것'이 아니라 '시험이 쉬워진 것'이다.
**같은 문장으로 예측을 비교하는 것**(마지막 셀)이 더 공정한 비교다.

이것이 이 문제에서 가장 배울 점일지도 모른다. **훈련 데이터의 오염은 성능 지표까지 오염시킨다.**

한편 **검증 데이터가 뒷부분 20%라는 점**도 짚어야 한다.
정제 전에는 **검증 데이터의 상당 부분이 라이선스 문서**다.
즉 정제 전 모델은 **소설로 배워서 법률 문서로 시험을 본 셈**이다.
정제 후에는 소설로 배워 소설로 시험을 본다.
그래서 두 모델의 검증 지표를 곧바로 견주는 것은 엄밀하지 않다. 채점 기준 자체가 다르기 때문이다.
**같은 문장으로 예측을 비교하는 것**(마지막 셀)이 더 공정한 비교다.

이것이 이 문제에서 가장 배울 점일지도 모른다. **훈련 데이터의 오염은 성능 지표까지 오염시킨다.**

### 문제 검토

- **적절성: 적합. 실무 감각을 기르는 문제다.** 지금까지 예제들은 모두 깔끔한 데이터를 썼는데,
  이 문제는 **현실의 데이터에는 쓰레기가 섞여 있다**는 것을 보여 준다. 소설 한 권을 그대로 내려받아 쓴
  본문 예제가 실은 정제되지 않은 데이터였다는 것을 뒤늦게 드러내는 구성도 좋다.
- **[검토] '예상해 본 후 확인하라'는 순서가 좋다.** 6-5, 6-7과 같은 구조라 장 전체의 흐름이 일관된다.
- **★ [검토] 검증 데이터도 함께 오염된다는 점을 짚어 주면 좋겠다.** 본문 예제는 원문을 앞뒤로 잘라
  검증 데이터를 만드는데, 정제하지 않으면 **검증 데이터의 상당 부분이 라이선스 문서**가 된다.
  그러면 정제 전후의 검증 손실을 비교하는 것 자체가 의미를 잃는다(채점 기준이 다르므로).
  이 점을 모르면 독자가 숫자를 잘못 해석할 수 있다.
- **[검토] '소설의 내용이 아닌 부분'을 어떻게 찾을지 힌트가 없다.** 구텐베르크 전자책이 `*** START OF ... ***`
  표시를 쓴다는 것을 모르면 눈으로 찾아 글자 수를 세야 한다. 각주 4가 프로젝트 구텐베르크를 소개하고 있으니,
  거기에 한 줄 덧붙이거나 지문에 힌트를 주면 좋겠다.
- **[검토] 성능이 크게 좋아지지는 않는다는 점도 열어 두어야 한다.** "결과를 확인해 보자"라고만 하면
  독자는 큰 개선을 기대하는데 실제로는 미미하다. 이것이 정상이라는 안내가 없으면 자기 구현을 의심한다.

**윤문안**

> **6-11** 소설 <오즈의 마법사> 원문 텍스트는 앞부분과 뒷부분에 소설의 내용이 아닌 다른 내용이 포함되어 있다
> (프로젝트 구텐베르크 전자책은 `*** START OF ... ***`와 `*** END OF ... ***` 사이가 본문이다).
> 이 부분이 띄어쓰기 모델의 성능에 어떤 영향을 미칠지 예상해 본 후, 소설의 내용이 아닌 부분을 제거하고
> 모델을 다시 학습해 결과를 확인해 보자. 이때 훈련 데이터뿐 아니라 검증 데이터도 함께 달라진다는 점을
> 염두에 두고, 같은 문장에 대한 두 모델의 예측 결과도 비교해 보자.

## 연습 문제 6-12

> `<pad>` 토큰을 사용해 입력 문자열 앞부분의 띄어쓰기도 예측할 수 있도록 데이터셋 클래스를 수정한 후
> `SpacingLSTM`을 학습하고 결과를 확인해 보자.

### 설계

본문 p26이 지적한 한계다.

> 이 모델은 입력 샘플의 마지막 글자 다음에 띄어쓰기가 있는지 판단하므로,
> 입력 샘플 길이가 25인 예제에서 **첫 24자 다음의 띄어쓰기는 예측할 수 없다.**

해법은 본문 p36의 그림 6-15가 보여 준다. **입력 문자열 앞에 `<pad>` 토큰을 24개 덧붙이면**
0번째 글자에 대해서도 `<pad>` 24개 + 그 글자로 길이 25의 윈도우를 만들 수 있다.

구현은 본문 p36이 말한 대로 **`__len__()`과 `__getitem__()` 두 메서드만 고치면 된다.**

- `__len__()`: `길이 - 윈도우 + 1`이 아니라 **문자열 길이 그대로**(모든 위치를 예측하므로)
- `__getitem__(idx)`: `idx`가 윈도우보다 앞이면 부족한 만큼 앞을 `<pad>`로 채운다

어휘 사전에도 `<pad>`를 등록해야 한다. **기존 글자의 번호를 바꾸지 않도록 맨 뒤에 붙인다.**

In [15]:
PAD_TOKEN = '<pad>'
pad_vocab = dict(clean_vocab)
pad_vocab[PAD_TOKEN] = len(pad_vocab)     # 기존 번호를 유지하려고 맨 뒤에 추가
PAD_IDX = pad_vocab[PAD_TOKEN]
print(f'어휘 사전 크기: {len(clean_vocab)} -> {len(pad_vocab)} (<pad> 번호 {PAD_IDX})')

class PaddedSpacingDataset(Dataset):
    """앞쪽에 <pad>를 채워 문자열의 모든 위치를 예측할 수 있게 한 데이터셋."""

    def __init__(self, input_sequence, target_sequence, vocab, sequence_length):
        self.vocab = vocab
        self.sequence_length = sequence_length
        self.pad_idx = vocab[PAD_TOKEN]
        self.input_idx = torch.tensor([vocab[char] for char in input_sequence])
        self.target_idx = torch.tensor([int(value) for value in target_sequence])

    def __len__(self):
        # 모든 위치를 예측하므로 문자열 길이만큼의 샘플이 만들어진다
        return len(self.input_idx)

    def __getitem__(self, idx):
        start = idx - self.sequence_length + 1
        if start < 0:
            # 앞이 모자라면 <pad>로 채운다
            pad = torch.full((-start,), self.pad_idx, dtype=torch.long)
            subsequence = torch.cat([pad, self.input_idx[: idx + 1]])
        else:
            subsequence = self.input_idx[start: idx + 1]
        x = F.one_hot(subsequence, num_classes=len(self.vocab)).float()
        return x, self.target_idx[idx]

# 앞쪽 샘플이 제대로 만들어지는지 확인한다
check_dataset = PaddedSpacingDataset(clean_input[:100], clean_target[:100], pad_vocab, SEQUENCE_LENGTH)
reversed_pad_vocab = {i: token for token, i in pad_vocab.items()}
for idx in (0, 1, 24, 25):
    x, y = check_dataset[idx]
    tokens = [reversed_pad_vocab[i] for i in x.argmax(dim=1).tolist()]
    shown = ''.join('_' if t == PAD_TOKEN else t for t in tokens)
    print(f'idx={idx:3d}: {shown!r} -> 정답 {y.item()}')

어휘 사전 크기: 46 -> 47 (<pad> 번호 46)
idx=  0: '________________________[' -> 정답 0
idx=  1: '_______________________[i' -> 정답 0
idx= 24: '[illustration]thewonderfu' -> 정답 0
idx= 25: 'illustration]thewonderful' -> 정답 1


In [16]:
pad_train_dataset = PaddedSpacingDataset(clean_input[:clean_split], clean_target[:clean_split],
                                        pad_vocab, SEQUENCE_LENGTH)
pad_valid_dataset = PaddedSpacingDataset(clean_input[clean_split:], clean_target[clean_split:],
                                        pad_vocab, SEQUENCE_LENGTH)
pad_train_loader = DataLoader(pad_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
pad_valid_loader = DataLoader(pad_valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'훈련 샘플 {len(pad_train_dataset):,}개 (패딩 없이 {len(clean_train_dataset):,}개)')

torch.manual_seed(SEED)
pad_model = SpacingLSTM(len(pad_vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print('<pad>를 추가한 데이터셋으로 SpacingLSTM 학습')
pad_result = train_with_early_stopping(pad_model, pad_train_loader, pad_valid_loader,
                                       monitor='loss', label='[패딩] ')

훈련 샘플 133,503개 (패딩 없이 133,479개)
<pad>를 추가한 데이터셋으로 SpacingLSTM 학습


  [패딩] 에포크  1 | 훈련 손실 0.3906 | 검증 손실 0.3095 | 정확도 85.94% | F1 0.6738


  [패딩] 에포크  2 | 훈련 손실 0.2697 | 검증 손실 0.2602 | 정확도 88.33% | F1 0.7627


  [패딩] 에포크  3 | 훈련 손실 0.2267 | 검증 손실 0.2307 | 정확도 89.83% | F1 0.7823


  [패딩] 에포크  4 | 훈련 손실 0.2002 | 검증 손실 0.2234 | 정확도 90.18% | F1 0.7962


  [패딩] 에포크  5 | 훈련 손실 0.1812 | 검증 손실 0.2162 | 정확도 90.62% | F1 0.8043


  [패딩] 에포크  6 | 훈련 손실 0.1640 | 검증 손실 0.2124 | 정확도 90.92% | F1 0.8036


  [패딩] 에포크  7 | 훈련 손실 0.1491 | 검증 손실 0.2184 | 정확도 90.95% | F1 0.8086


  [패딩] 에포크  8 | 훈련 손실 0.1333 | 검증 손실 0.2260 | 정확도 90.90% | F1 0.8058


  [패딩] 에포크  9 | 훈련 손실 0.1188 | 검증 손실 0.2335 | 정확도 90.90% | F1 0.8024
  조기 종료: 에포크 9 (최적 에포크 6, 기준 loss)


In [17]:
def make_spacing_padded(model, text, model_vocab, sequence_length=SEQUENCE_LENGTH):
    """첫 글자부터 모든 위치의 띄어쓰기를 예측한다."""
    stripped = ''.join(char for char in text.strip().lower() if not char.isspace())
    stripped = ''.join(char for char in stripped if char in model_vocab)
    dataset = PaddedSpacingDataset(stripped, '0' * len(stripped), model_vocab, sequence_length)
    model.eval()
    result = []
    with torch.no_grad():
        for i in range(len(dataset)):
            x, _ = dataset[i]
            predicted = model(x.unsqueeze(0).to(device)).argmax().item()
            result.append(stripped[i])
            if predicted == 1:
                result.append(' ')
    return ''.join(result)

for sample in [SAMPLE, 'I want to make a cool stuff, so I study deep learning.']:
    print(f'원문       : {sample.strip().lower()}')
    print(f'패딩 없음  : {make_spacing(clean_model, sample, clean_vocab)}')
    print(f'패딩 사용  : {make_spacing_padded(pad_model, sample, pad_vocab)}')
    print()

원문       : besides, her castle stands on the edge of the desert, so she may know a way to cross it.
패딩 없음  : besides,hercastlestandsonthe edgeof the desert, so she may knowaway to cross it . 
패딩 사용  : besides, he rcastlestandson the edgeof the desert, so she may knowaway to crossit . 

원문       : i want to make a cool stuff, so i study deep learning.
패딩 없음  : iwanttomakeacoolstuff,soi study deeplearning . 
패딩 사용  : iwantto make acoolstuff, sois tudy deeplearning . 



### 풀이 해설

**무엇이 달라졌는가**

가장 눈에 띄는 변화는 **문자열 앞부분이다.** 패딩이 없는 모델은 첫 24자를 손대지 못해
`besides,hercastlestandson the edge of...`처럼 **앞부분이 통째로 붙어 나온다.**
패딩을 쓴 모델은 **첫 글자부터 띄어쓰기를 넣는다.**

샘플 수도 늘었다. 길이 `N`인 문자열에서 `N - 24`개가 아니라 `N`개의 샘플이 만들어지므로,
**앞쪽 24개 위치에 대한 학습 기회가 새로 생긴다.**

**`<pad>`를 어휘 사전 맨 뒤에 붙인 이유**

앞에 넣으면 기존 글자의 고유 번호가 전부 하나씩 밀린다. 그러면 이전에 학습한 모델의 파라미터를
재사용할 수 없고, 저장해 둔 데이터셋도 다시 만들어야 한다.
**맨 뒤에 붙이면 기존 번호가 그대로 유지**되어 영향이 국소적이다.
실무에서 특수 토큰을 맨 앞에 두는 관행도 있지만(그때는 처음부터 자리를 비워 둔다),
**이미 만들어진 사전에 나중에 더할 때는 뒤가 안전하다.**

**남는 한계**

패딩은 앞부분 문제를 해결하지만 **본문 p36이 말한 더 근본적인 문제는 그대로다.**
이 모델은 여전히 **예측 위치의 앞쪽만** 본다. 그래서 문장 부호 앞의 띄어쓰기 오류
(`deep learning .`처럼 마침표 앞을 띄우는 실수)는 줄어들지 않는다.
바로 다음 글자가 마침표라는 사실을 볼 수 없기 때문이다.

이를 해결하려면 p36~37의 추가 설명이 말한 대로 **예측 위치를 가운데로 옮기거나 양방향 순환 신경망**을 써야 한다.
즉 **`<pad>` 도입은 필요조건이지 충분조건이 아니다.**
재미있는 것은 예측 위치를 가운데로 옮기는 방법도 **앞뒤 양쪽에 `<pad>`가 필요**하다는 점이다.
이 문제에서 만든 데이터셋이 그 확장의 발판이 된다.

### 문제 검토

- **적절성: 적합.** 본문 p26이 제기한 한계('첫 24자는 예측할 수 없다')를 p36에서 그림 6-15로 해법만 보여 주고
  넘어가는데, 이 문제가 그 사이를 메운다. **본문에서 제기한 문제를 연습 문제에서 해결하는** 좋은 구조다.
- **[검토] 본문이 구현 범위까지 알려 준 것이 친절하다.** p36의 "데이터셋 클래스의 `__len__()`과 `__getitem__()`
  메서드만 수정하면 적용할 수 있다"가 정확한 안내라, 독자가 어디를 고칠지 헤매지 않는다.
- **[검토] 어휘 사전에 `<pad>`를 추가해야 한다는 점이 지문에 없다.** `<pad>` 토큰을 쓰려면 사전에 등록해야 하고,
  그러면 `input_size`가 1 늘어 모델도 다시 만들어야 한다. 본문 p8이 "사용할 토큰은 어휘 사전에도 함께 등록해야
  한다"고 미리 말해 두었으니 큰 함정은 아니지만, 형태 불일치 오류로 한 번 막힐 가능성은 있다.
- **★ [검토] '결과를 확인해 보자'의 확인 방법이 모호하다.** 검증 손실만 보면 패딩 모델이 더 나빠 보일 수 있다.
  앞쪽 24개 위치는 정보가 적어 맞히기 어려운 샘플인데, 그런 샘플이 새로 들어왔기 때문이다.
  **이 문제의 성과는 지표가 아니라 '앞부분도 띄어 쓴다'는 사실에 있으므로**, 예측 결과를 직접 보게 해야 한다.
- **[검토] 남는 한계를 짚게 하면 좋겠다.** 패딩으로도 문장 부호 앞 오류는 해결되지 않는데,
  이를 확인하면 p36~37의 '예측 위치 옮기기'와 '양방향 순환 신경망'이 왜 필요한지가 분명해진다.

**윤문안**

> **6-12** `<pad>` 토큰을 사용해 입력 문자열 앞부분의 띄어쓰기도 예측할 수 있도록 데이터셋 클래스를 수정한 후
> `SpacingLSTM`을 학습하고 결과를 확인해 보자. 이때 어휘 사전에도 `<pad>` 토큰을 등록해야 한다.
> 학습한 모델로 문장의 띄어쓰기를 예측해 앞부분이 어떻게 달라졌는지 직접 비교해 보고,
> 이 방법으로도 해결되지 않는 오류가 남아 있는지 살펴보자.

## 연습 문제 6-13

> LSTM과 함께 소개한 GRU 계층을 사용해 띄어쓰기 모델을 정의하고 학습해 보자.
> LSTM을 사용한 모델과 어떤 차이가 있는지를 학습 시간까지 포함해 검토해 보자.

In [18]:
torch.manual_seed(SEED)
gru_model = SpacingGRU(len(clean_vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print('SpacingGRU 학습')
gru_result = train_with_early_stopping(gru_model, clean_train_loader, clean_valid_loader,
                                       monitor='loss', label='[GRU] ')

torch.manual_seed(SEED)
rnn_model = SpacingRNN(len(clean_vocab), HIDDEN_SIZE, NUM_LAYERS, 2)
print()
print('(비교) SpacingRNN 학습')
rnn_result = train_with_early_stopping(rnn_model, clean_train_loader, clean_valid_loader,
                                       monitor='loss', label='[RNN] ')

SpacingGRU 학습


  [GRU] 에포크  1 | 훈련 손실 0.3655 | 검증 손실 0.3010 | 정확도 85.99% | F1 0.6662


  [GRU] 에포크  2 | 훈련 손실 0.2598 | 검증 손실 0.2444 | 정확도 89.48% | F1 0.7671


  [GRU] 에포크  3 | 훈련 손실 0.2197 | 검증 손실 0.2263 | 정확도 90.16% | F1 0.7947


  [GRU] 에포크  4 | 훈련 손실 0.1937 | 검증 손실 0.2163 | 정확도 90.56% | F1 0.8000


...

  [RNN] 에포크  8 | 훈련 손실 0.2173 | 검증 손실 0.2506 | 정확도 89.35% | F1 0.7797


  [RNN] 에포크  9 | 훈련 손실 0.2073 | 검증 손실 0.2318 | 정확도 89.85% | F1 0.7818


  [RNN] 에포크 10 | 훈련 손실 0.2005 | 검증 손실 0.2271 | 정확도 90.37% | F1 0.7956


  [RNN] 에포크 11 | 훈련 손실 0.1935 | 검증 손실 0.2208 | 정확도 90.46% | F1 0.7935


  [RNN] 에포크 12 | 훈련 손실 0.1882 | 검증 손실 0.2255 | 정확도 90.35% | F1 0.7955


In [19]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print(f'{"모델":>12} {"파라미터 수":>13} {"최적 에포크":>11} {"검증 손실":>10} {"정확도":>9} {"F1":>9} {"에포크당 시간":>14}')
print('-' * 92)
for name, model, result in [('단순 RNN', rnn_model, rnn_result),
                            ('LSTM', clean_model, clean_result),
                            ('GRU', gru_model, gru_result)]:
    best = result['history'][result['best_epoch'] - 1]
    per_epoch = result['elapsed'] / result['last_epoch']
    print(f'{name:>12} {count_parameters(model):13,d} {result["best_epoch"]:11d} '
          f'{best[2]:10.4f} {best[3] * 100:8.2f}% {best[4]:9.4f} {per_epoch:13.1f}초')

print()
print(f'원문     : {SAMPLE.lower()}')
for name, model in [('단순 RNN', rnn_model), ('LSTM', clean_model), ('GRU', gru_model)]:
    print(f'{name:<9}: {make_spacing(model, SAMPLE, clean_vocab)}')

          모델        파라미터 수      최적 에포크      검증 손실       정확도        F1        에포크당 시간
--------------------------------------------------------------------------------------------
      단순 RNN       209,922          11     0.2208    90.46%    0.7935           5.8초
        LSTM       838,146           6     0.2087    90.85%    0.8100           7.2초
         GRU       628,738           5     0.2112    90.92%    0.8040           7.2초

원문     : besides, her castle stands on the edge of the desert, so she may know a way to cross it.
단순 RNN   : besides,hercastlestandsonthe edgeof the desert, so she may know away to cross it. 
LSTM     : besides,hercastlestandsonthe edgeof the desert, so she may knowaway to cross it . 
GRU      : besides,hercastlestandsonthe edgeof the desert, so she may know away to cross it. 


### 풀이 해설

**구현은 한 글자 차이다.** 본문 p34가 말한 대로 `nn.LSTM`을 `nn.GRU`로 바꾸기만 하면 된다.
생성자 인자(`input_size`, `hidden_size`, `num_layers`, `batch_first`)가 같고,
첫 번째 출력 텐서의 형태도 `(B, S, hidden_size)`로 같기 때문이다.

다만 **두 번째 반환값은 셋이 모두 다르다.**

| 계층 | 두 번째 반환값 |
|---|---|
| `nn.RNN` | 마지막 숨겨진 상태 `h` |
| `nn.LSTM` | 숨겨진 상태와 셀 상태의 쌍 `(h, c)` |
| `nn.GRU` | 마지막 숨겨진 상태 `h` (GRU는 셀 상태가 없다) |

이 예제처럼 **첫 번째 출력만 쓰면** 세 계층이 그대로 호환된다. 본문 p31이 짚은 그대로다.

**파라미터 수가 세 계층의 성격을 말해 준다.** 위 표에서 확인할 수 있듯,
같은 `hidden_size`와 `num_layers`에서 **단순 RNN : GRU : LSTM ≈ 1 : 3 : 4**의 비율이다.
학습 가능한 부품의 수가 각각 1개, 3개, 4개이기 때문이다.

- 단순 RNN: 상태 갱신 1개
- GRU: 리셋 게이트, 갱신 게이트, 후보 상태 = 3개
- LSTM: 망각·입력·출력 게이트, 장기 기억 후보 = 4개

실제로 측정한 결과다.

| 모델 | 파라미터 수 | 최적 에포크 | 검증 손실 | 정확도 | F1 | 에포크당 시간 |
|---|---|---|---|---|---|---|
| 단순 RNN | 209,922 | 11 | 0.2208 | 90.46% | 0.7935 | 5.8초 |
| LSTM | 838,146 | 6 | 0.2087 | 90.85% | 0.8100 | 7.2초 |
| GRU | 628,738 | 5 | 0.2112 | **90.92%** | 0.8040 | 7.2초 |

파라미터 수가 **209,922 : 628,738 : 838,146 ≈ 1 : 3 : 4**로, 부품 수와 정확히 맞아떨어진다.

**그런데 에포크당 시간은 예상과 다르다.** 파라미터가 25% 적은 GRU가 LSTM과 **똑같이 7.2초**다.
본문 추가 설명의 "LSTM보다 가볍고 학습도 빠른 편"이 이 실험에서는 드러나지 않았다.

왜 그럴까? **GPU에서는 파라미터 수가 곧 시간이 아니기 때문**이다.
파이토치는 `nn.LSTM`과 `nn.GRU`를 모두 cuDNN의 최적화된 커널로 실행하는데,
순환 신경망의 병목은 행렬 연산량이 아니라 **시점을 하나씩 순서대로 처리해야 하는 구조**에 있다.
25 시점을 차례로 도는 비용이 지배적이라, 한 시점 안의 계산량 차이는 잘 드러나지 않는다.
CPU에서 돌리거나 배치를 크게 잡으면 차이가 보이기 시작한다.

**대신 GRU는 다른 곳에서 이득을 본다.** 최적 에포크가 5로 LSTM(6)보다 빨리 수렴했다.
에포크당 시간이 같아도 **전체 학습 시간은 GRU가 짧다.** 그리고 모델 크기는 25% 작다.

**성능은 어떤가.** 검증 손실은 LSTM이, 정확도는 GRU가 근소하게 앞선다. 사실상 같다고 보는 것이 맞다.
단순 RNN은 파라미터가 4분의 1인데도 F1 0.7935로 크게 뒤지지 않는데,
이 예제의 윈도우가 25자로 비교적 짧아 장기 의존성 문제가 심하지 않기 때문으로 보인다.
다만 **최적 에포크가 11로 가장 늦고** 검증 손실도 가장 높다.

**정리하면 GRU는 '더 작은 모델로 비슷한 성능'을 낸다.** 속도 이득은 실행 환경에 따라 달라지므로
기대하지 말고 재 보아야 한다. 어느 쪽이 나을지는 데이터마다 다르니 **둘 다 해 보는 것이 정답**이고,
구현이 한 글자 차이이니 실제로 그렇게 하는 것이 어렵지 않다.

### 문제 검토

- **적절성: 적합.** 본문 추가 설명('LSTM만큼 좋지만 더 가벼운 GRU')이 GRU를 소개만 하고 넘어가는데,
  이 문제가 그것을 실제로 확인하게 한다. 구현 부담이 거의 없어(한 글자) 실행 장벽도 낮다.
- **★ [검토] '학습 시간까지 포함해'라는 조건은 좋지만, 기대한 결과가 나오지 않을 수 있다.**
  본문 추가 설명은 GRU가 "LSTM보다 가볍고 학습도 빠른 편"이라고 하는데, **GPU에서 재면 에포크당 시간이 같게 나온다**
  (위 실행 결과에서 둘 다 7.2초). 파이토치가 두 계층을 모두 cuDNN 커널로 실행하고,
  순환 신경망의 병목이 연산량이 아니라 시점을 순서대로 도는 구조에 있기 때문이다.
  독자가 "왜 안 빨라지지?" 하고 자기 측정을 의심할 수 있으므로, **전체 학습 시간(수렴까지 걸린 에포크 × 시간)으로
  비교하게 하거나, 환경에 따라 차이가 작을 수 있다는 단서를 달면** 좋겠다.
  실제로 GRU는 5 에포크, LSTM은 6 에포크에 수렴해 전체 시간은 GRU가 짧다.
- **[검토] 파라미터 수도 함께 세어 보게 하면 좋겠다.** 학습 시간은 실행 환경에 따라 들쭉날쭉하지만,
  **파라미터 수는 누가 세어도 같다.** 단순 RNN : GRU : LSTM = 1 : 3 : 4라는 비율이 나오는데,
  이는 각 계층의 부품 수(1개, 3개, 4개)와 정확히 일치해 구조를 이해하는 데 도움이 된다.
- **[검토] 단순 RNN까지 세 모델을 비교하게 하면 더 좋다.** LSTM과 GRU만 견주면 둘 다 잘해서
  '게이트가 있으면 좋다'는 결론이 흐릿해진다. 단순 RNN을 넣으면 세 계층의 위치가 한눈에 정리된다.

**윤문안**

> **6-13** LSTM과 함께 소개한 GRU 계층을 사용해 띄어쓰기 모델을 정의하고 학습해 보자.
> 단순 RNN, LSTM 모델과 어떤 차이가 있는지를 파라미터 수와 학습 시간까지 포함해 검토해 보자.

## 연습 문제 6-14 [도전 문제]

> 소설 <오즈의 마법사>를 학습해 <오즈의 마법사> 스타일의 소설을 쓰는 모델을 만들어 보자.
> 6-2절과 6-3절의 예제 코드를 참고해 만들면 되는데, 어휘 사전을 구성하는 토큰의 단위(글자 또는 어절 등)나
> 하이퍼파라미터 등의 조건을 바꿔 보며 다양한 모델을 만들어 보거나 마중물 텍스트를 바꿔 가며 생성해 보자.

### 설계

두 절의 코드를 섞으면 된다.

| 가져올 것 | 어디서 |
|---|---|
| 다음 토큰을 예측하는 구조(출력 크기 = 어휘 사전 크기) | 6-2절 `MelodyRNN` |
| 자기회귀 생성 함수 | 6-2절 `generate_sequence()` |
| LSTM 계층, 소설 텍스트 처리, 장치 이동 | 6-3절 `SpacingLSTM` |

6-3절의 띄어쓰기 모델과 **딱 한 가지가 다르다.** 띄어쓰기 모델은 출력이 2개(있다/없다)였지만,
생성 모델은 **출력이 어휘 사전 크기만큼** 필요하다. 다음에 올 글자를 고르는 분류 문제이기 때문이다.

그리고 이번에는 **띄어쓰기를 제거하지 않는다.** 공백도 하나의 토큰으로 학습해야 읽을 수 있는 글이 나온다.

In [20]:
# 소설 본문을 글자 단위로 학습한다. 이번에는 공백도 토큰으로 포함한다
gen_text = re.sub(r'\s+', ' ', novel_text.lower()).strip()
gen_vocab = {char: i for i, char in enumerate(sorted(set(gen_text)))}
gen_reversed_vocab = {i: char for char, i in gen_vocab.items()}
GEN_SEQUENCE_LENGTH = 40      # 생성 모델은 더 긴 문맥이 필요하다
print(f'학습 텍스트: {len(gen_text):,}자, 어휘 사전 크기: {len(gen_vocab)}')

class GenerationDataset(Dataset):
    """앞 sequence_length개 글자로 다음 글자를 맞히는 데이터셋."""

    def __init__(self, text, vocab, sequence_length):
        self.vocab = vocab
        self.sequence_length = sequence_length
        self.idx = torch.tensor([vocab[char] for char in text])

    def __len__(self):
        return len(self.idx) - self.sequence_length

    def __getitem__(self, idx):
        window = self.idx[idx: idx + self.sequence_length + 1]
        x = F.one_hot(window[:-1], num_classes=len(self.vocab)).float()
        return x, window[-1]

gen_dataset = GenerationDataset(gen_text, gen_vocab, GEN_SEQUENCE_LENGTH)
gen_loader = DataLoader(gen_dataset, batch_size=256, shuffle=True)
print(f'샘플 수: {len(gen_dataset):,}개')

학습 텍스트: 206,527자, 어휘 사전 크기: 47
샘플 수: 206,487개


In [21]:
class NovelLSTM(nn.Module):
    """6-2절의 MelodyRNN 구조에 LSTM 계층을 넣은 생성 모델."""

    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size=vocab_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)     # 출력 크기 = 어휘 사전 크기

    def forward(self, x):
        outputs, _ = self.lstm(x)
        return self.fc(outputs[:, -1, :])

torch.manual_seed(SEED)
novel_model = NovelLSTM(len(gen_vocab), 256, 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(novel_model.parameters(), lr=LEARNING_RATE)

GEN_EPOCHS = 6
print('NovelLSTM 학습')
for epoch in range(1, GEN_EPOCHS + 1):
    started = time.time()
    train_loss = train_epoch(novel_model, gen_loader, criterion, optimizer, device)
    print(f'  에포크 {epoch}/{GEN_EPOCHS} | 훈련 손실 {train_loss:.4f} | {time.time() - started:.0f}초')

NovelLSTM 학습


  에포크 1/6 | 훈련 손실 2.2534 | 8초


  에포크 2/6 | 훈련 손실 1.6425 | 11초


  에포크 3/6 | 훈련 손실 1.4092 | 7초


  에포크 4/6 | 훈련 손실 1.2707 | 7초


  에포크 5/6 | 훈련 손실 1.1769 | 7초


  에포크 6/6 | 훈련 손실 1.1043 | 9초


In [22]:
@torch.no_grad()
def generate_text(model, primer, length, vocab, reversed_vocab, temperature=1.0,
                  sequence_length=GEN_SEQUENCE_LENGTH):
    """자기회귀 방식으로 글자를 이어 붙여 생성한다.

    temperature 가 0이면 항상 최고 점수를 고르고(본문 [코드 6-8] 방식),
    값이 클수록 무작위성이 커진다.
    """
    model.eval()
    text = primer.lower()
    text = ''.join(char for char in text if char in vocab)
    for _ in range(length):
        window = text[-sequence_length:]
        if len(window) < sequence_length:              # 마중물이 짧으면 공백으로 채운다
            window = ' ' * (sequence_length - len(window)) + window
        idx = torch.tensor([vocab[char] for char in window])
        x = F.one_hot(idx, num_classes=len(vocab)).float().unsqueeze(0).to(device)
        logits = model(x)[0]
        if temperature <= 0:
            next_idx = logits.argmax().item()
        else:
            probabilities = torch.softmax(logits / temperature, dim=0)
            next_idx = torch.multinomial(probabilities, 1).item()
        text += reversed_vocab[next_idx]
    return text

PRIMER = 'dorothy lived in the midst of the great kansas prairies '
for temperature in (0.0, 0.5, 1.0):
    label = '항상 최고 점수(본문 방식)' if temperature == 0 else f'temperature={temperature}'
    generated = generate_text(novel_model, PRIMER, 300, gen_vocab, gen_reversed_vocab, temperature)
    print(f'[{label}]')
    print(generated)
    print()

[항상 최고 점수(본문 방식)]
dorothy lived in the midst of the great kansas prairies and said, “where is the same and said to the emerald city, and the scarecrow was a great way to the tin woodman and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the scarecrow and the 



[temperature=0.5]
dorothy lived in the midst of the great kansas prairies a leg and was a strange and the scarecrow and the scarecrow was not all the fell of the bed, and the winkies, and said, “why do you seek me a lovely and well as easily.” “where see a country,” said the scarecrow, “and i will be a coward,” replied the lion. “where dorothy was a great of yours all a r



[temperature=1.0]
dorothy lived in the midst of the great kansas prairies ard he had armed him—inly on thes a great of good in the little, only that the going was preatly on, alston’ great didlyou always to qunter a great; for with you.” “wears could,” remarked. “dor’t pyeans therefter of thy oft is place, the awery,” caplied the scarecrow, “for i suppose no one i shall n



### 풀이 해설

**6-2절과 6-3절을 어떻게 합치는가**

구조는 6-2절의 `MelodyRNN`과 같다. 다음 토큰을 예측하는 분류 모델을 만들고,
예측을 입력에 이어 붙여 반복 호출하는 **자기회귀** 방식으로 생성한다.

달라지는 것은 세 가지다.

1. **계층을 LSTM으로 바꾼다.** 소설은 멜로디보다 훨씬 긴 문맥이 필요하다.
2. **윈도우를 크게 잡는다.** 멜로디는 4음이면 됐지만 글은 40자 정도는 봐야 단어와 문장 구조가 잡힌다.
3. **공백을 어휘 사전에 포함한다.** 6-3절에서는 공백을 제거했지만, 생성 모델에서는 공백도 만들어 내야 한다.

출력 크기가 `2`(띄어쓰기 여부)에서 **어휘 사전 크기**로 바뀌는 것도 중요한 차이다.

**생성 결과 읽기**

학습을 몇 에포크만 해도 **영어 단어처럼 생긴 문자열**이 나온다.
글자 단위 모델이 `the`, `and`, `dorothy` 같은 단어를 만들어 내고 마침표 뒤에 공백을 넣는다는 것은,
**모델이 철자법과 띄어쓰기를 데이터에서 스스로 배웠다**는 뜻이다. 누구도 알려 준 적이 없다.

물론 문장의 의미는 이어지지 않는다. 40자 윈도우로는 문장 하나를 겨우 담으므로
**문단 수준의 일관성은 원리적으로 불가능하다.**
이 한계를 넘어서는 것이 9장 이후의 어텐션과 트랜스포머다.

**`temperature`가 바꾸는 것**

본문 [코드 6-8]은 늘 최고 점수의 토큰을 고른다(`argmax`). 이 방식은 **같은 마중물에 늘 같은 결과**를 내고,
본문 p17이 지적했듯 짧은 패턴을 무한 반복하기 쉽다. 위 실행 결과의 첫 번째가 그렇다.

대신 **로짓을 확률로 바꿔 뽑으면**(`torch.multinomial`) 매번 다른 결과가 나온다.
`temperature`는 그 확률 분포를 얼마나 평평하게 만들지를 정한다.

| `temperature` | 성격 |
|---|---|
| 0에 가까움 | `argmax`와 같다. 안전하지만 반복에 빠진다 |
| 0.5 정도 | 그럴듯하면서도 반복을 피한다. 대체로 가장 읽을 만하다 |
| 1.0 이상 | 다양하지만 철자가 무너지기 시작한다 |

**이것이 이 문제에서 가장 배울 점**일 수 있다. 본문은 `argmax` 방식만 보여 주는데,
실제 생성 모델은 거의 예외 없이 확률적 샘플링을 쓴다. 생성의 다양성은 **모델이 아니라 뽑는 방법**에서 나온다.

**조건을 바꿔 볼 만한 것들**

- **토큰 단위**: 어절 단위로 바꾸면 어휘 사전이 수천 개로 커지지만 문법은 더 그럴듯해진다.
  대신 처음 보는 단어를 만들 수 없고 `<unk>` 처리가 필요하다(연습 문제 6-1의 표 참고).
- **윈도우 크기**: 40 → 80으로 늘리면 문장 사이의 연결이 좋아지지만 학습 시간이 비례해 늘어난다.
- **`hidden_size`, `num_layers`**: 6-7에서 확인했듯 크다고 좋아지지는 않는다. 데이터 크기에 맞춰야 한다.
- **에포크 수**: 생성 모델은 과적합이 '원문 그대로 재생'으로 나타난다. 6-5에서 본 것과 같다.

### 문제 검토

- **적절성: 6장을 마무리하는 도전 문제로 매우 적합하다.** 6-2절(생성)과 6-3절(LSTM, 소설 데이터)을
  **처음으로 합치게 한다.** 두 절을 따로 배운 독자가 '이 둘이 이어지는구나'를 깨닫는 자리다.
  그리고 결과물이 눈에 보이는 글이라 성취감도 크다.
- **[검토] 열린 문제로서 방향 제시가 적절하다.** "토큰의 단위나 하이퍼파라미터 등의 조건을 바꿔 보며"라는
  안내가 있어 막막하지 않으면서도 답을 정해 주지 않는다. 도전 문제의 좋은 본보기다.
- **★ [검토] 출력 크기가 바뀐다는 점을 짚어 주면 좋겠다.** 6-3절의 `SpacingLSTM`을 그대로 가져오면
  **출력이 2개(띄어쓰기 여부)**라 생성에 쓸 수 없다. 출력 크기를 어휘 사전 크기로 바꿔야 하는데,
  이것이 '판단하는 모델'과 '생성하는 모델'을 가르는 결정적 차이다.
  독자가 스스로 발견하면 가장 좋지만, 못 찾으면 한참 헤맨다. **이 문제에서 가장 중요한 설계 판단**이므로
  물음으로 던져 두는 것도 방법이다.
- **★ [검토] 생성 방법(샘플링)에 대한 언급이 있으면 좋겠다.** 본문 [코드 6-8]의 `argmax` 방식을 그대로 쓰면
  **같은 문구가 끝없이 반복되는 결과**가 나온다. 본문 p17이 그 원인을 이미 설명해 두었으니
  "반복을 줄이려면 어떻게 해야 할까"를 물어 확률적 샘플링으로 이끌 수 있다.
  실제 생성 모델은 거의 모두 이 방식을 쓰므로 알아 둘 값어치가 크다.
- **[검토] 학습 시간에 대한 안내가 있으면 좋겠다.** 소설 전체를 글자 단위로 학습하면 샘플이 18만 개가 넘어,
  실습 환경에 따라 시간이 꽤 걸린다. 본문 p25가 띄어쓰기 모델에서 "가능하면 GPU 등의 별도 연산 장치를
  사용하는 편이 좋다"고 일러 준 것처럼, 여기서도 한마디 있으면 친절하다.

**윤문안**

> **6-14** [도전 문제] 소설 <오즈의 마법사>를 학습해 <오즈의 마법사> 스타일의 소설을 쓰는 모델을 만들어 보자.
> 6-2절과 6-3절의 예제 코드를 참고해 만들면 되는데, 띄어쓰기 모델과 달리 **출력 크기를 무엇으로 해야 할지**
> 먼저 생각해 보자. 그리고 어휘 사전을 구성하는 토큰의 단위(글자 또는 어절 등)나 하이퍼파라미터 등의 조건을
> 바꿔 보며 다양한 모델을 만들어 보거나 마중물 텍스트를 바꿔 가며 생성해 보자.
> 생성 결과에 같은 문구가 반복된다면, [코드 6-8]처럼 늘 가장 점수가 높은 토큰을 고르는 대신
> 어떤 방법을 쓸 수 있을지도 생각해 보자.